In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets,transforms, models
import torchvision
import pandas as pd

import math
import random
import time
import numpy as np
from typing import Optional
import os
import imageio
import time
import warnings
import sys
import copy
import json
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

# Deterministic seed control (used by MyTrainer/data loaders)
RANDOM_SEED = 999
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# Generators for DataLoader reproducibility
general_generator = torch.Generator()
general_generator.manual_seed(RANDOM_SEED)
train_generator = torch.Generator()
train_generator.manual_seed(RANDOM_SEED)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print('PyTorch', torch.__version__, 'CUDA available:', torch.cuda.is_available())

PyTorch 2.1.2 CUDA available: True


## Model definitions
The notebook contains the same classes as in your supplied code: `SpectralGatingExpansion`, `SpatioTemporalBlock`, and `ThermRNet`.

In [ ]:
class SpectralGatingExpansion(nn.Module):
    def __init__(self, dim, expansion_factor=2):
        super().__init__()
        hidden_dim = int(dim * expansion_factor)
        self.project_in = nn.Conv1d(dim, hidden_dim, kernel_size=1)
        self.act = nn.GELU()
        # complex weights stored as real/imag pairs, keep as float32 param
        # self.complex_weight = nn.Parameter(torch.randn(hidden_dim, freq_dim, 2, dtype=torch.float32) * 0.02)
        self.complex_weight = nn.Parameter(torch.randn(hidden_dim, 1, 2, dtype=torch.float32) * 0.02)
        self.project_out = nn.Conv1d(hidden_dim, dim, kernel_size=1)

    def forward(self, x):
        # x shape: (B*H*W, D, T) or (B, D, T)
        # Preserve original dtype and ensure FFT / complex ops run in full precision
        orig_dtype = x.dtype
        B, D, T = x.shape
        x = self.project_in(x)
        x = self.act(x)
        # Disable autocast (mixed-precision) for FFT and complex multiplication to avoid cuFFT half-precision limitations
        with torch.cuda.amp.autocast(enabled=False):
            x_fp32 = x.float()
            x_fft = torch.fft.rfft(x_fp32, n=T, dim=-1, norm='ortho')
            weight = torch.view_as_complex(self.complex_weight).to(x_fft.dtype)
            x_fft = x_fft * weight
            x_ifft = torch.fft.irfft(x_fft, n=T, dim=-1, norm='ortho')
            x = x_ifft.to(orig_dtype)
        x = self.project_out(x)
        return x


class SpatioTemporalBlock(nn.Module):
    def __init__(self, dim, expansion_factor=2):
        super().__init__()
        self.spatial_conv = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim),
            nn.BatchNorm2d(dim),
            nn.Conv2d(dim, dim, kernel_size=1),
            nn.GELU()
        )
        self.temporal_spectral = SpectralGatingExpansion(dim, expansion_factor)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        # Input: (B, D, T, H, W)
        B, D, T, H, W = x.shape
        x_s = x.permute(0, 2, 1, 3, 4).reshape(B * T, D, H, W)
        x_s = self.spatial_conv(x_s)
        x_s = x_s.view(B, T, D, H, W).permute(0, 2, 1, 3, 4)
        x = x + x_s
        x_t = x.permute(0, 3, 4, 1, 2).reshape(B * H * W, D, T)
        x_t = self.temporal_spectral(x_t)
        x_t = x_t.view(B, H, W, D, T).permute(0, 3, 4, 1, 2)
        x = x + x_t
        return x

class ThermRNet(nn.Module):
    def __init__(self, in_channels=3, base_dim=64, num_classes=2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(in_channels, base_dim, kernel_size=(1, 5, 5), stride=(1, 2, 2), padding=(0, 2, 2)),
            nn.BatchNorm3d(base_dim),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))
        )
        self.stage1 = nn.Sequential(
            SpatioTemporalBlock(base_dim),
            SpatioTemporalBlock(base_dim)
        )
        self.downsample = nn.Conv3d(base_dim, base_dim * 2, kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))
        self.stage2 = nn.Sequential(
            SpatioTemporalBlock(base_dim * 2),
            SpatioTemporalBlock(base_dim * 2),
            SpatioTemporalBlock(base_dim * 2)
        )
        self.final_dim = base_dim * 2
        self.classifier = nn.Sequential(
            nn.Conv1d(self.final_dim, self.final_dim, 1),
            nn.Dropout(0.3),
            nn.ReLU(),
            nn.Conv1d(self.final_dim, num_classes, 1)
        )

    def forward(self, x):
        # x: (B, 3, T, 72, 72)
        x = self.stem(x)
        x = self.stage1(x)
        x = self.downsample(x)
        x = self.stage2(x)
        x_pool = torch.mean(x, dim=(3, 4))  # (B, final_dim, T)
        logits = self.classifier(x_pool)   # (B, num_classes, T)
        return logits

## Synthetic dataset and dataloader

In [12]:
from torchvision import transforms
from torchvision.transforms.functional import to_pil_image, to_tensor
import imageio

mean = [0.5, 0.5, 0.5]
std = [0.5, 0.5, 0.5]

# Use a top-level function so it can be pickled by multiprocessing on Windows
def div255(x):
    return x / 255.0

transform = transforms.Compose([  # 数据增强
    transforms.Lambda(div255),  # 替代 ToTensor 的归一化
    transforms.RandomRotation(10),
    transforms.Resize(96, antialias=True),
    transforms.RandomCrop(72),
    transforms.RandomHorizontalFlip(),
    transforms.Normalize(mean=mean, std=std),
])

transform_valid = transforms.Compose([
    transforms.Lambda(div255),  # 替代 ToTensor 的归一化
    transforms.Resize(72, antialias=True),
    transforms.Normalize(mean=mean, std=std)  # 然后进行标准化
])


def standardized_data(data):
    """Z-score standardization for video data."""
    data = data - np.mean(data)
    data = data / np.std(data)
    data[np.isnan(data)] = 0
    return data


def apply_transform_to_video(video, transform):
    """
    Apply the same transformation to every frame of the video.

    Args:
        video (np.ndarray): Video with shape (T, Height, Width, channels).
        transform (callable): Transform function that takes PIL image and returns transformed PIL image.

    Returns:
        torch.Tensor: Transformed video with shape (channels, T, Height, Width).
    """
    assert isinstance(video, np.ndarray)
    video_tensor = torch.from_numpy(video).permute(0, 3, 1, 2).float() # (T, C, H, W)
    # The transforms we use expect images in CHW or PIL; we apply transforms per-frame via stack if needed
    # Here we assume `transform` can operate on tensors directly (our transforms use Lambda / Normalize).
    transformed = transform(video_tensor)  # transform should support (T, C, H, W) or tensor transforms
    return transformed.permute(1, 0, 2, 3)  # (C, T, H, W)

class MyCustomDataset(Dataset):
    def __init__(self, dataset_name, dataset_dir, file_path_list_path=None, transform=None):
        self.dataset_name = dataset_name
        self.dataset_dir = dataset_dir
        self.file_path_list_path = file_path_list_path
        self.transform = transform
        self.data_list = os.listdir(dataset_dir)
        self.inputs = [os.path.join(dataset_dir, x) for x in self.data_list if 'input' in x]
        self.labels = [os.path.join(dataset_dir, x) for x in self.data_list if 'label' in x]
        self.inputs.sort()
        self.labels.sort()

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data = np.load(self.inputs[index])  # 获取一个分段数据
        label = np.load(self.labels[index])
        if random.random() < 0.1:   # 数据增强
            indices = list(range(len(data)))
            random.shuffle(indices)
            data = data[indices]
            label = label[indices]
        data = apply_transform_to_video(data, self.transform)  # 转换后每个像素值变为(0, 1)
        # Ensure torch tensors with desired shapes and dtypes

        # data shape: (C, T, H, W) -> convert to (C, T, H, W) already
        
        label = torch.from_numpy(label).long()
        filename = chunk_id = 1
        return data, label, filename, chunk_id

class MyCustomTestDataset(Dataset):
    def __init__(self, dataset_name, dataset_dir, file_path_list_path=None, transform=None):
        self.dataset_name = dataset_name
        self.dataset_dir = dataset_dir
        self.file_path_list_path = file_path_list_path
        self.transform = transform
        self.data_list = os.listdir(dataset_dir)
        self.inputs = [os.path.join(dataset_dir, x) for x in self.data_list if 'input' in x]
        self.labels = [os.path.join(dataset_dir, x) for x in self.data_list if 'label' in x]
        self.inputs.sort()
        self.labels.sort()

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data = np.load(self.inputs[index])  # 获取一个分段数据
        label = np.load(self.labels[index])
        data = apply_transform_to_video(data, self.transform)

        
        label = torch.from_numpy(label).long()
        filename = chunk_id = 1
        return data, label, filename, chunk_id

## Training / Validation functions

In [13]:
def _unpack_batch(batch):
    # Accept either (video, labels) or (video, labels, filename, chunk_id)
    if isinstance(batch, (list, tuple)):
        if len(batch) >= 2:
            vids = batch[0]
            labels = batch[1]
        else:
            vids, labels = batch
    else:
        vids, labels = batch
    return vids, labels


def train_one_epoch(model, loader, optimizer, device, loss_fn, scaler=None):
    model.train()
    total_loss = 0.0
    total_frames = 0
    correct = 0
    for batch in loader:
        vids, labels = _unpack_batch(batch)
        vids = vids.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(vids)
                B, C, T = logits.shape
                logits_flat = logits.permute(0, 2, 1).reshape(B * T, C)
                labels_flat = labels.view(-1)
                loss = loss_fn(logits_flat, labels_flat)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(vids)
            B, C, T = logits.shape
            logits_flat = logits.permute(0, 2, 1).reshape(B * T, C)
            labels_flat = labels.view(-1)
            loss = loss_fn(logits_flat, labels_flat)
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * B * T
        total_frames += B * T
        preds = logits_flat.argmax(dim=1)
        correct += (preds == labels_flat).sum().item()

    return total_loss / total_frames, correct / total_frames


def validate(model, loader, device, loss_fn):
    model.eval()
    total_loss = 0.0
    total_frames = 0
    correct = 0
    with torch.no_grad():
        for batch in loader:
            vids, labels = _unpack_batch(batch)
            vids = vids.to(device)
            labels = labels.to(device)
            logits = model(vids)
            B, C, T = logits.shape
            logits_flat = logits.permute(0, 2, 1).reshape(B * T, C)
            labels_flat = labels.view(-1)
            loss = loss_fn(logits_flat, labels_flat)
            total_loss += loss.item() * B * T
            total_frames += B * T
            preds = logits_flat.argmax(dim=1)
            correct += (preds == labels_flat).sum().item()

    return total_loss / total_frames, correct / total_frames

## Dry-run / quick training example
Execute this cell to instantiate the model on your available device and run a single forward pass (or a couple of training epochs) on synthetic data. Adjust `BATCH` and `FRAMES` if you need smaller memory footprint.

In [14]:
# Configuration
BATCH = 1
FRAMES = 32
EPOCHS = 1
BASE_DIM = 64
NUM_CLASSES = 2
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device', DEVICE)

# Instantiate model
model = ThermRNet(in_channels=3, base_dim=BASE_DIM, num_classes=NUM_CLASSES).to(DEVICE)
print('Model params (M):', sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6)


# Optionally run one training epoch (uncomment to use)
# optimizer = optim.AdamW(model.parameters(), lr=1e-3)
# loss_fn = nn.CrossEntropyLoss()
# model.train()
# for vids, labels in train_loader:
#     vids = vids.to(DEVICE); labels = labels.to(DEVICE)
#     optimizer.zero_grad()
#     logits = model(vids) # (B, C, T)
#     B, C, T = logits.shape
#     logits_flat = logits.permute(0, 2, 1).reshape(B*T, C)
#     labels_flat = labels.view(-1)
#     loss = loss_fn(logits_flat, labels_flat)
#     loss.backward(); optimizer.step()
#     print('train loss', loss.item())

Using device cuda
Model params (M): 0.393602


In [15]:
# Trainer class adapted from `code_train.ipynb`
import os
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import OneCycleLR
from tqdm import tqdm

class MyTrainer:
    def __init__(self, model, device, train_loader, val_loader=None, *, epochs=10, max_lr=1e-3, weight_decay=1e-2, save_dir='checkpoints'):
        self.model = model
        self.device = device
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs
        self.max_lr = max_lr
        self.weight_decay = weight_decay
        self.save_dir = save_dir
        os.makedirs(self.save_dir, exist_ok=True)

        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.AdamW(self.model.parameters(), lr=max_lr, weight_decay=self.weight_decay)
        # Scheduler will be created lazily when steps_per_epoch is known
        self.scheduler = None
        self.scaler = GradScaler()

    def _maybe_create_scheduler(self):
        if self.scheduler is None:
            steps_per_epoch = max(1, len(self.train_loader))
            self.scheduler = OneCycleLR(self.optimizer, max_lr=self.max_lr, epochs=self.epochs, steps_per_epoch=steps_per_epoch)

    def _unpack_batch(self, batch):
        # Accept batches of form (inputs, targets) or (inputs, targets, filename, chunk_id)
        if isinstance(batch, (list, tuple)) and len(batch) >= 2:
            inputs = batch[0]
            targets = batch[1]
        else:
            raise ValueError('Unexpected batch format')
        return inputs, targets

    def train_one_epoch(self, epoch):
        self.model.train()
        self._maybe_create_scheduler()
        running_loss = 0.0
        correct = 0
        total = 0
        pbar = tqdm(self.train_loader, desc=f"Train epoch {epoch+1}/{self.epochs}")
        for batch in pbar:
            inputs, targets = self._unpack_batch(batch)
            inputs = inputs.to(self.device)
            # targets may have shape (N, T) or (N,) depending on dataset; we pick appropriate handling
            # if isinstance(targets, (list, tuple)):
            #     targets = torch.stack([t.to(self.device) for t in targets])
            # else:
            targets = targets.to(self.device)

            self.optimizer.zero_grad()
            with autocast():
                outputs = self.model(inputs)
                # If outputs is (N, C, ...) and targets are per-frame, try to reduce appropriately.
                if outputs.dim() > 2 and targets.dim() > 1:
                    # Attempt to use last-dim logits -> assume classification per-frame along dim=1
                    # Convert outputs to (N, C, T) or (N, C) depending on model
                    out = outputs
                    # If outputs shape (N, C, T) and targets (N, T) -> flatten
                    if out.dim() == 3 and targets.dim() == 2:
                        N, C, T = out.shape
                        out_flat = out.permute(0, 2, 1).reshape(N*T, C)
                        tgt_flat = targets.reshape(N*T)
                        loss = self.criterion(out_flat, tgt_flat)
                        pred = out_flat.argmax(dim=1)
                        correct += (pred == tgt_flat).sum().item()
                        total += tgt_flat.numel()
                    else:
                        # fallback: try simple cross-entropy on 2D outputs
                        loss = self.criterion(out, targets.view(-1))
                        pred = out.argmax(dim=1)
                        correct += (pred == targets.view(-1)).sum().item()
                        total += targets.numel()
                else:
                    loss = self.criterion(outputs, targets)
                    pred = outputs.argmax(dim=1)
                    correct += (pred == targets).sum().item()
                    total += targets.numel()

            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()
            if self.scheduler is not None:
                try:
                    self.scheduler.step()
                except Exception:
                    pass

            running_loss += loss.item()
            pbar.set_postfix({'loss': running_loss / (len(pbar) if len(pbar) else 1), 'acc': 100. * correct / max(1, total)})

        epoch_loss = running_loss / max(1, len(self.train_loader))
        epoch_acc = 100. * correct / max(1, total)
        return epoch_loss, epoch_acc

    def validate(self):
        if self.val_loader is None:
            return None, None
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc='Validation'):
                inputs, targets = self._unpack_batch(batch)
                inputs = inputs.to(self.device)
                targets = targets.to(self.device)
                outputs = self.model(inputs)

                if outputs.dim() > 2 and targets.dim() > 1:
                    N, C, T = outputs.shape
                    out_flat = outputs.permute(0, 2, 1).reshape(N*T, C)
                    tgt_flat = targets.reshape(N*T)
                    loss = self.criterion(out_flat, tgt_flat)
                    pred = out_flat.argmax(dim=1)
                    correct += (pred == tgt_flat).sum().item()
                    total += tgt_flat.numel()
                else:
                    loss = self.criterion(outputs, targets)
                    pred = outputs.argmax(dim=1)
                    correct += (pred == targets).sum().item()
                    total += targets.numel()

                running_loss += loss.item()

        val_loss = running_loss / max(1, len(self.val_loader))
        val_acc = 100. * correct / max(1, total)
        return val_loss, val_acc

    def save_model(self, name='model_best.pth'):
        path = os.path.join(self.save_dir, name)
        torch.save({'model_state_dict': self.model.state_dict(), 'optimizer_state_dict': self.optimizer.state_dict()}, path)
        return path

    def load_model(self, path):
        ckpt = torch.load(path, map_location=self.device)
        self.model.load_state_dict(ckpt['model_state_dict'])
        if 'optimizer_state_dict' in ckpt:
            try:
                self.optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            except Exception:
                pass

    def fit(self):
        best_val = math.inf
        for epoch in range(self.epochs):
            train_loss, train_acc = self.train_one_epoch(epoch)
            val_loss, val_acc = self.validate()
            print(f"Epoch {epoch+1}/{self.epochs} - train_loss: {train_loss:.4f} train_acc: {train_acc:.2f}% val_loss: {val_loss} val_acc: {val_acc}")
            # save best
            if val_loss is not None and val_loss < best_val:
                best_val = val_loss
                self.save_model(f'best_epoch_{epoch+1}.pth')


## Next steps / Tips
- Replace `SyntheticVideoDataset` with your real data loader that returns `(video, frame_labels)`.
- If you prefer to run the `.py` script version (already added to the repo), use the script `train_deepspecnet72.py` for larger training runs and checkpointing.
- To conserve GPU memory, decrease `BATCH`, set smaller `FRAMES`, or reduce `BASE_DIM`.

In [16]:
import time
import torch
from torch.utils.data import Dataset, DataLoader

# Ensure model/trainer are present in the kernel
if 'ThermRNet' not in globals() or 'MyTrainer' not in globals():
    raise RuntimeError('ThermRNet or MyTrainer not defined in kernel. Run the model and trainer cells first.')

# Quick synthetic dataset + smoke test (num_workers=0 to avoid Windows worker issues)
class SyntheticVideoDatasetSmall(Dataset):
    def __init__(self, num_samples=8, T=16, H=72, W=72, channels=3, num_classes=2):
        self.num_samples = num_samples
        self.T = T
        self.H = H
        self.W = W
        self.C = channels
        self.num_classes = num_classes

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Return video as (C, T, H, W) and per-frame labels (T,)
        video = torch.rand(self.C, self.T, self.H, self.W, dtype=torch.float32)
        labels = torch.randint(0, self.num_classes, (self.T,), dtype=torch.long)
        return video, labels

print('Creating synthetic dataset and running trainer.train_one_epoch with num_workers=0')
# small test loader
synth_ds = SyntheticVideoDatasetSmall(num_samples=8, T=16, H=72, W=72, channels=3, num_classes=(NUM_CLASSES if 'NUM_CLASSES' in globals() else 2))
synth_loader = DataLoader(synth_ds, batch_size=2, shuffle=True, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device for test:', device)
# small model to reduce compute
test_model = ThermRNet(in_channels=3, base_dim=32, num_classes=(NUM_CLASSES if 'NUM_CLASSES' in globals() else 2)).to(device)
print('Test model params (M):', sum(p.numel() for p in test_model.parameters() if p.requires_grad)/1e6)

test_trainer = MyTrainer(test_model, device, synth_loader, None, epochs=1, max_lr=1e-3)
start = time.time()
train_loss, train_acc = test_trainer.train_one_epoch(0)
end = time.time()
print(f'Synthetic training finished in {end-start:.2f}s — loss: {train_loss:.6f}, acc: {train_acc:.4f}')

Creating synthetic dataset and running trainer.train_one_epoch with num_workers=0
Device for test: cuda
Test model params (M): 0.102594


Train epoch 1/1: 100%|██████████| 4/4 [00:00<00:00, 14.93it/s, loss=0.686, acc=57]  

Synthetic training finished in 0.27s — loss: 0.685673, acc: 57.0312


In [17]:
# ------------------ Create dataloaders and train (real dataset preferred) ------------------
# Set your dataset paths (from code_train.ipynb). Update these to your local dataset paths if needed.
train_dataset_dir = r'X:\my_thermal_nose_dataset\RawData_CHUNK160_RESIZE128_littleROI_TRAIN\mydataset'
valid_dataset_dir = r'X:\my_thermal_nose_dataset\RawData_CHUNK160_RESIZE128_littleROI_TEST\mydataset'

TRAIN_BATCH = 2
TEST_BATCH = 2
FRAMES = 160
EPOCHS = 50
BASE_DIM = 64
NUM_CLASSES = 2
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device', DEVICE)

use_real_dataset = os.path.isdir(train_dataset_dir) and os.path.isdir(valid_dataset_dir)
if use_real_dataset:
    print('Found dataset directories, will use MyCustomDataset')
    train_ds = MyCustomDataset(dataset_name='train', dataset_dir=train_dataset_dir, transform=transform)
    val_ds = MyCustomTestDataset(dataset_name='valid', dataset_dir=valid_dataset_dir, transform=transform_valid)
    train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=TEST_BATCH, shuffle=False, num_workers=0)
else:
    print('Dataset directories not found. Falling back to synthetic dataset for testing.')
    train_ds = SyntheticVideoDataset(num_samples=40, T=FRAMES, H=72, W=72, channels=3, num_classes=NUM_CLASSES)
    val_ds = SyntheticVideoDataset(num_samples=8, T=FRAMES, H=72, W=72, channels=3, num_classes=NUM_CLASSES)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False)

# Instantiate model
model = ThermRNet(in_channels=3, base_dim=BASE_DIM, num_classes=NUM_CLASSES).to(DEVICE)
print('Model params (M):', sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6)

optimizer = optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler() if DEVICE == 'cuda' else None
trainer = MyTrainer(model, DEVICE, train_loader, val_loader if 'val_loader' in globals() else None, epochs=EPOCHS if 'EPOCHS' in globals() else 10, max_lr=1e-3)

Using device cuda
Found dataset directories, will use MyCustomDataset
Model params (M): 0.393602


In [18]:
# trainer.train_one_epoch(1)

In [19]:
# ---- Run full training and save checkpoints ----
# Ensure trainer, optimizer, dataloaders exist in the kernel
if 'trainer' not in globals():
    raise RuntimeError('`trainer` not found in kernel. Run previous cells to create trainer and dataloaders first.')
# Set / confirm epochs
EPOCHS = EPOCHS if 'EPOCHS' in globals() else 2
trainer.epochs = EPOCHS
print(f"Starting training for {trainer.epochs} epochs. Checkpoints will be saved to: {trainer.save_dir}")
# Run training (this will save the best model during training using trainer.save_model inside fit)
trainer.fit()
# After training, save the final model weights explicitly
final_name = f'final_model_epoch{trainer.epochs}.pth'
final_path = trainer.save_model(final_name)
print('Final model saved to:', final_path)

Starting training for 50 epochs. Checkpoints will be saved to: checkpoints


Validation: 100%|██████████| 38/38 [00:17<00:00,  2.13it/s]



Epoch 1/50 - train_loss: 0.3804 train_acc: 84.65% val_loss: 0.488970228323811 val_acc: 80.725


Validation: 100%|██████████| 38/38 [00:17<00:00,  2.15it/s]



Epoch 2/50 - train_loss: 0.2701 train_acc: 89.76% val_loss: 0.3077807449980786 val_acc: 87.14166666666667


...

Epoch 48/50 - train_loss: 0.0918 train_acc: 96.34% val_loss: 0.13550232110643073 val_acc: 94.89166666666667


Validation: 100%|██████████| 38/38 [00:17<00:00,  2.15it/s]



Epoch 49/50 - train_loss: 0.0891 train_acc: 96.47% val_loss: 0.13049567990789288 val_acc: 95.075


Validation: 100%|██████████| 38/38 [00:17<00:00,  2.16it/s]

Epoch 50/50 - train_loss: 0.0901 train_acc: 96.39% val_loss: 0.14114104721106982 val_acc: 94.70833333333333
Final model saved to: checkpoints\final_model_epoch50.pth
